**[Run this NB in Google Colab](https://colab.research.google.com/github/snad-space/coniferest/blob/master/docs/notebooks/onnx_serialization.ipynb)**

# ONNX model snapshots

This notebook shows how to use the `SaveToOnnx` callback to periodically save the model of a running `Session` to ONNX format, and how to load the saved ONNX model back and use it to score new data.

In [ ]:
## Install and import the required libraries

This notebook requires `onnxruntime`. Install it if it isn't already available:

```bash
pip install onnxruntime
```

In [ ]:
import numpy as np
import onnxruntime as rt
import os
from coniferest.datasets import single_outlier
from coniferest.isoforest import IsolationForest
from coniferest.session import Session
from coniferest.session.callback import Label, SaveToOnnx, TerminateAfter

## Prepare a small dataset

We use the `single_outlier` toy dataset that ships with coniferest.

In [ ]:
data, metadata = single_outlier()
data = data.astype(np.float32)

print(f"Data shape: {data.shape}")

## Run a session that saves the model to ONNX

We attach `SaveToOnnx` as an `on_decision_callback`. Here we save every 5 decisions, keeping a numbered file for each save (`overwrite=False`), so we can inspect how the model evolves over time.

In [ ]:
ONNX_DIR = "onnx_models"
EXPERT_BUDGET = 20


def decision(index, x, session):
    # Non-interactive: pick a fixed answer just to drive the demo session
    return Label.REGULAR


model = IsolationForest(
    n_trees=100,
    random_seed=0,
)

session = Session(
    data=data,
    metadata=np.arange(len(data)),
    model=model,
    decision_callback=decision,
    on_decision_callbacks=[
        TerminateAfter(EXPERT_BUDGET),
        SaveToOnnx(
            directory=ONNX_DIR,
            filename="model.onnx",
            every_n_decisions=5,
            overwrite=False,
        ),
    ],
)

session.model.fit(data)
session.run()

Let's see which ONNX snapshots were saved during the session.

In [ ]:
saved_files = sorted(os.listdir(ONNX_DIR))
print(saved_files)

## Load a saved ONNX model and use it

Pick the last saved snapshot and load it with `onnxruntime` to run inference, without needing the original coniferest model object.

In [ ]:
last_snapshot_path = os.path.join(ONNX_DIR, saved_files[-1])

sess = rt.InferenceSession(last_snapshot_path)
input_name = sess.get_inputs()[0].name
label_name = "score"

onnx_scores = sess.run([label_name], {input_name: data})[0].reshape(-1)

print(f"Top 5 most anomalous scores (ONNX): {np.sort(onnx_scores)[:5]}")

## Overwrite strategy example

If you only care about the latest model and don't want to keep every snapshot, use `overwrite=True` (the default) so the callback always writes to the same file.

In [ ]:
OVERWRITE_DIR = "onnx_model_latest"

callback = SaveToOnnx(directory=OVERWRITE_DIR, filename="model.onnx")

session_overwrite = Session(
    data=data,
    metadata=np.arange(len(data)),
    model=IsolationForest(n_trees=100, random_seed=0),
    decision_callback=decision,
    on_decision_callbacks=[
        TerminateAfter(EXPERT_BUDGET),
        callback,
    ],
)
session_overwrite.model.fit(data)
session_overwrite.run()

print(os.listdir(OVERWRITE_DIR))

## Saving the whole session with PickleSession
Unlike `SaveToOnnx`, which only saves the model, `PickleSession` saves the entire session object (model, known labels, current candidate, scores, etc.) using Python's `pickle`. This means an interrupted session can be fully resumed later, not just the model.

In [ ]:
from coniferest.session.callback import PickleSession
import pickle

PICKLE_DIR = "session_snapshots"

session_with_pickle = Session(
    data=data,
    metadata=np.arange(len(data)),
    model=IsolationForest(n_trees=100, random_seed=0),
    decision_callback=decision,
    on_decision_callbacks=[
        TerminateAfter(EXPERT_BUDGET),
        PickleSession(directory=PICKLE_DIR, every_n_decisions=5),
    ],
)
session_with_pickle.model.fit(data)
session_with_pickle.run()

print(os.listdir(PICKLE_DIR))

Now let's load the saved session back and confirm it's fully resumed, including the known labels.

In [ ]:
saved_sessions = sorted(os.listdir(PICKLE_DIR))
last_session_path = os.path.join(PICKLE_DIR, saved_sessions[-1])

with open(last_session_path, "rb") as f:
    restored_session = pickle.load(f)

print(f"Known labels: {restored_session.known_labels}")
print(f"Model type: {type(restored_session.model)}")